In [1]:
# Import Packages
import pandas as pd
import re

# Import from local files
from data_cleaning import DataCleaner
from extraction import extract_data
from entity_resolution import EntityResolver

# Set pandas display options
pd.set_option("display.width", 1000)
pd.set_option("display.max_columns", None)


# ----------------------------------------------------------
# Extract
# ----------------------------------------------------------

extracted_df = extract_data(
    "2025 Roll - Parcels Export.xlsx",
    "Only Delinquent Parcels"
)


# ----------------------------------------------------------
# Clean
# ----------------------------------------------------------

cleaner = DataCleaner()

clean_df = cleaner.clean_data(
    extracted_df
)


# ----------------------------------------------------------
# Entity Resolution
# ----------------------------------------------------------

(
    entity_occurrences,
    entities,
    parcel_entity_roles,
    association_evidence,
    entity_associations
) = EntityResolver().build(
    clean_df
)

# Create audit dataframe.
resolution_audit = (
    entity_occurrences
    .merge(
        entities[
            [
                "entity_id",
                "canonical_name",
                "canonical_address",
                "entity_type",
                "occurrence_count"
            ]
        ],
        on="entity_id",
        how="left",
        validate="many_to_one",
        suffixes=("_source", "_final")
    )
)

resolution_audit.head()

,occurrence_id,parcel_number,entity_name,associated_address,role,entity_type_source,organization_type,organization_classification_confidence,address_source,entity_id,canonical_name,canonical_address,entity_type_final,occurrence_count
0,1,01-01-201-013,POLOMCAK CLAUDIA,"31029 30TH ST, PAW PAW, MI 49079",Owner,Person,NaN,NaN,owner_address,3141,POLOMCAK CLAUDIA,"31029 30TH ST, PAW PAW, MI 49079",Person,2
1,2,01-01-251-013,CLAWSON DORRINDA &,"10591 N 12TH ST, PLAINWELL, MI 49080",Owner,Person,NaN,NaN,owner_address,883,CLAWSON DORRINDA &,"10591 N 12TH ST, PLAINWELL, MI 49080",Person,2
2,3,01-01-251-050,CLAWSON DORRINDA L,"10591 N 12TH ST, PLAINWELL, MI 49080",Owner,Person,NaN,NaN,owner_address,884,CLAWSON DORRINDA L,"10591 N 12TH ST, PLAINWELL, MI 49080",Person,6
3,4,01-01-251-060,CLAWSON DORRINDA L,"10591 N 12TH ST, PLAINWELL, MI 49080",Owner,Person,NaN,NaN,owner_address,884,CLAWSON DORRINDA L,"10591 N 12TH ST, PLAINWELL, MI 49080",Person,6
4,5,01-01-301-050,SPIEKERMANN TIM & JUDY,"10238 N 10TH ST, PLAINWELL, MI 49080",Owner,Person,NaN,NaN,owner_address,3727,SPIEKERMANN TIM & JUDY,"10238 N 10TH ST, PLAINWELL, MI 49080",Person,2


## Entities with the highest occurrence_count

In [2]:
entities.sort_values(
    "occurrence_count",
    ascending=False
).head(10)

,entity_id,canonical_name,entity_type,canonical_address,organization_type,organization_classification_confidence,occurrence_count
2058,2059,KAYLOR HOLDINGS LLC,Organization,"1405 HOLIDAY LN, PORTAGE, MI 49024",LLC,High,62
577,578,"BROOKS, SAMUEL C",Person,"804 CHARLOTTE AVE, KALAMAZOO, MI 49048",NaN,NaN,42
3754,3755,STATE WIDE RENTALS LLC,Organization,"606 BRYANT ST, KALAMAZOO, MI 49001",LLC,High,25
20,21,"19 PROPS, LLC",Organization,"200 W MICHIGAN AVE STE 201, KALAMAZOO, MI 49007",LLC,High,21
1489,1490,GIBSON ROBERT M & MARTHA M,Person,"8713 S 38TH ST, SCOTTS, MI 49088",NaN,NaN,18
4162,4163,WALLY BROWN LLC,Organization,"563 TREASURE ISLAND DR, MATTAWAN, MI 49071",LLC,High,18
2959,2960,OSWALT JOHN & LINDA TRUST,Organization,"15168 S 37TH ST, VICKSBURG, MI 49097",Trust,Medium,16
423,424,"BISHOP, BRADLEY J",Person,"PO BOX 312, PORTAGE, MI 49081",NaN,NaN,16
4015,4016,UJAMAA REALTY HOLDINGS LLC,Organization,"160 W CAMINO REAL UNIT #676, BOCA RATON, FL 33432",LLC,High,15
1738,1739,"HERTEL, JOSEPH",Person,"2326 S ROSE ST, KALAMAZOO, MI 49001",NaN,NaN,15


## Entities with canonical names containing a middle name or middle initial

In [19]:
SUFFIXES = {
    "JR",
    "SR",
    "JUNIOR",
    "SENIOR",
    "II",
    "III",
    "IV",
}


def get_middle_tokens(name):
    """
    Return possible middle-name tokens from a person name.

    Supports both:
        SMITH, JOHN A
        SMITH JOHN A

    This is for manual inspection only, not entity-resolution logic.
    """

    if pd.isna(name):
        return tuple()

    name = str(name).upper().strip()
    name = name.replace(".", "")

    # Skip obvious joint/compound records
    if (
        "&" in name
        or "/" in name
        or re.search(r"\bAND\b", name)
    ):
        return tuple()

    if "," in name:

        _, given_part = name.split(",", 1)
        tokens = given_part.strip().split()

        if tokens and tokens[-1] in SUFFIXES:
            tokens = tokens[:-1]

        # FIRST [MIDDLE...]
        return tuple(tokens[1:])

    else:

        tokens = name.split()

        if tokens and tokens[-1] in SUFFIXES:
            tokens = tokens[:-1]

        # Assume LAST FIRST [MIDDLE...]
        if len(tokens) < 3:
            return tuple()

        return tuple(tokens[2:])

people_with_middle_names = (
    entities[
        entities["entity_type"] == "Person"
    ]
    .copy()
)

people_with_middle_names["middle_tokens"] = (
    people_with_middle_names[
        "canonical_name"
    ]
    .apply(get_middle_tokens)
)

people_with_middle_names = (
    people_with_middle_names[
        people_with_middle_names[
            "middle_tokens"
        ].apply(len) > 0
    ]
    .sort_values(
        "occurrence_count",
        ascending=False
    )
)

people_with_middle_names[
    [
        "entity_id",
        "canonical_name",
        "middle_tokens",
        "canonical_address",
        "occurrence_count"
    ]
]

,entity_id,canonical_name,middle_tokens,canonical_address,occurrence_count
577,578,"BROOKS, SAMUEL C","(C,)","804 CHARLOTTE AVE, KALAMAZOO, MI 49048",42
423,424,"BISHOP, BRADLEY J","(J,)","PO BOX 312, PORTAGE, MI 49081",16
3673,3674,"SMITH, SEAN A","(A,)","5093 W GH AVE, KALAMAZOO, MI 49009",15
4424,4425,"YOUNG, CAVEL J","(J,)","2111 N DRAKE RD, KALAMAZOO, MI 49006",14
3186,3187,"PRIMER, DANIEL L JR","(L,)","6841 N SPRINKLE RD, KALAMAZOO, MI 49004",12
...,...,...,...,...,...
3259,3260,"RAY, PENNY LOU","(LOU,)","709 COLLINS ST, KALAMAZOO, MI 49001",1
3257,3258,RAY TYLER WAYNE,"(WAYNE,)","1131 N 30TH ST, GALESBURG, MI 49053-8739",1
3255,3256,RAY BRENDA LYNN,"(LYNN,)","1161 N 30TH ST, GALESBURG, MI 49053-8739",1
3254,3255,RAY BRENDA LYNN,"(LYNN,)","1131 N 30TH ST, GALESBURG, MI 49053-8739",1


- Some names have commas while others don't
    - For example, KAVANAUGH, JAMES J vs. CLAWSON DORRINDA L
        - Note that both had their middle initial listed correctly under middle_tokens
- The canonical name FRITH, TERRANCE JEROME A produced the middle_token (JEROME, A)
- The canonical name BROWN, ROBERT WENDELL, JR. produced the middle_token (WENDELL,,)
    - Note that there are two commas rather than the usual one
- GREATER TRUE WORSHIP COGIC produced (WORSHIP, COGIC)
- BLACKSTAR STABILITY REVITALIZATION produced (REVITALIZATION,)
- EDWARDS, KARRIE; EDWARDS, KENTON produced (EDWARDS,, KENTON)

Unrelated case:
There are two entities with the name WIDGER JAMES LEROY, one with the address 214 MARSHALL ST, ALLEGAN, MI 49010 and the other with the address 214 MARSHALL ST, ALLEGAN, MI 49010. Because they have the exact same name and their address, with the removal of ST from the first case, are the same - they should clearly be merged.

### Distinguishing middle initials from full middle names

In [13]:
def middle_type(tokens):

    if not tokens:
        return None

    if all(len(token) == 1 for token in tokens):
        return "Initial"

    return "Full/Other"


people_with_middle_names["middle_type"] = (
    people_with_middle_names[
        "middle_tokens"
    ]
    .apply(middle_type)
)

people_with_middle_names[
    "middle_type"
].value_counts()

middle_type
Initial       803
Full/Other    421
Name: count, dtype: int64

Out of 3,955 unique entity names, 1,240 show as containing a middle name or initial.

In [5]:
797 + 443

1240

## Entities the previously had a blank address

In [14]:
blank_address_cases = (
    resolution_audit
    .groupby(
        [
            "entity_id",
            "canonical_name",
            "canonical_address"
        ],
        dropna=False
    )
    .agg(
        had_blank_address=(
            "associated_address",
            lambda x: (
                x.fillna("")
                .str.strip()
                .eq("")
                .any()
            )
        ),
        source_addresses=(
            "associated_address",
            lambda x: tuple(
                sorted(
                    {
                        address
                        for address in x.fillna("")
                        if address != ""
                    }
                )
            )
        ),
        source_names=(
            "entity_name",
            lambda x: tuple(
                sorted(set(x))
            )
        ),
        occurrence_count=(
            "occurrence_id",
            "size"
        )
    )
    .reset_index()
)

blank_address_cases = (
    blank_address_cases[
        blank_address_cases["had_blank_address"]
        &
        (
            blank_address_cases[
                "canonical_address"
            ] != ""
        )
    ]
)

print(blank_address_cases.shape)
blank_address_cases.head(50)

(1580, 7)


,entity_id,canonical_name,canonical_address,had_blank_address,source_addresses,source_names,occurrence_count
15,16,1733 W. MAIN ST LLC,"1733 1/2 W MAIN ST, KALAMAZOO, MI 49006",True,"(1733 1/2 W MAIN ST, KALAMAZOO, MI 49006,)","(1733 W. MAIN ST LLC,)",2
19,20,19 PROPS LLC,"200 W MICHIGAN AVE STE 201, KALAMAZOO, MI 49007-",True,"(200 W MICHIGAN AVE STE 201, KALAMAZOO, MI 490...","(19 PROPS LLC,)",2
21,22,"1930 WEST MAIN, LLC","3260 NILES RD STE B, SAINT JOSEPH, MI 49085",True,"(3260 NILES RD STE B, SAINT JOSEPH, MI 49085,)","(1930 WEST MAIN, LLC,)",2
24,25,2219 HUMPHREY LLC,"100 W MICHIGAN AVE STE 200, KALAMAZOO, MI 49007",True,"(100 W MICHIGAN AVE STE 200, KALAMAZOO, MI 490...","(2219 HUMPHREY LLC,)",2
26,27,232 GRAND LLC,"11670 RAMSEYER DR, PLAINWELL, MI 49080",True,"(11670 RAMSEYER DR, PLAINWELL, MI 49080,)","(232 GRAND LLC,)",2
28,29,26TH ST PROPERTIES LLC,"2915 E GRAND RIVER AVE, HOWELL, MI 48843",True,"(2915 E GRAND RIVER AVE, HOWELL, MI 48843,)","(26TH ST PROPERTIES LLC,)",2
38,39,3030 S 9TH LLC,"PO BOX 312098, NEW BRAUNFELS, TX 78131",True,"(PO BOX 312098, NEW BRAUNFELS, TX 78131,)","(3030 S 9TH LLC,)",2
41,42,3381 FOREST VIEW INVESTORS LLC,"2265 LIVERNOIS STE 500, TROY, MI 48083",True,"(2265 LIVERNOIS STE 500, TROY, MI 48083,)","(3381 FOREST VIEW INVESTORS LLC,)",4
42,43,"3501 LAKE HOLDINGS, LLC","3501 LAKE ST, KALAMAZOO, MI 49048",True,"(3501 LAKE ST, KALAMAZOO, MI 49048,)","(3501 LAKE HOLDINGS, LLC,)",2
49,50,5752 TRUST,"4426 HUGH HOWELL RD # B-369, TUCKER, GA 30084",True,"(4426 HUGH HOWELL RD # B-369, TUCKER, GA 30084,)","(5752 TRUST,)",2


There are 1,580 cases of entities that used to have a blank address
- **QUESTION: Is the main thing to look for here that the source_names and canonical_name are in fact similar enough that we would want to merge them?**

## Names affected by CORP/CORPORATION, INC/INCORPORATED, CO/COMPANY

In [15]:
ENTITY_NAME_REPLACEMENTS = {
    "CORP": "CORPORATION",
    "INC": "INCORPORATED",
    "CO": "COMPANY"
}


def expand_entity_abbreviations(name):

    if pd.isna(name):
        return ""

    name = str(name).upper().strip()

    name = name.replace(".", "")

    name = re.sub(r"\s+", " ", name)
    name = re.sub(r"\s*,\s*", ", ", name)

    for old, new in ENTITY_NAME_REPLACEMENTS.items():

        name = re.sub(
            rf"\b{re.escape(old)}\b",
            new,
            name
        )

    return name.strip()

abbreviation_audit = resolution_audit.copy()

abbreviation_audit["expanded_name"] = (
    abbreviation_audit[
        "entity_name"
    ]
    .apply(expand_entity_abbreviations)
)

abbreviation_cases = (
    abbreviation_audit
    .groupby(
        [
            "entity_id",
            "canonical_name",
            "canonical_address"
        ]
    )
    .agg(
        source_names=(
            "entity_name",
            lambda x: tuple(
                sorted(set(x))
            )
        ),
        expanded_names=(
            "expanded_name",
            lambda x: tuple(
                sorted(set(x))
            )
        ),
        source_name_count=(
            "entity_name",
            "nunique"
        ),
        expanded_name_count=(
            "expanded_name",
            "nunique"
        )
    )
    .reset_index()
)

abbreviation_cases = (
    abbreviation_cases[
        (abbreviation_cases["source_name_count"] > 1)
        &
        (abbreviation_cases["expanded_name_count"] == 1)
    ]
)

abbreviation_cases.head(50)

,entity_id,canonical_name,canonical_address,source_names,expanded_names,source_name_count,expanded_name_count
110,111,AIM CORPORATION,"218 GLADYS ST, PORTAGE, MI 49002","(AIM CORP, AIM CORPORATION)","(AIM CORPORATION,)",2,1
1506,1507,"GILLEYLEN, MARVIN D SR.& B.A.","1315 N CHURCH ST, KALAMAZOO, MI 49007","(GILLEYLEN, MARVIN D SR& B.A, GILLEYLEN, MARVI...","(GILLEYLEN, MARVIN D SR& BA,)",2,1
1893,1894,J. SAJE LLC,"2900 KENSINGTON DR, KALAMAZOO, MI 49008","(J SAJE LLC, J. SAJE LLC)","(J SAJE LLC,)",2,1
2418,2419,MAIN QSR INC.,"1946 45TH ST STE C, MUNSTER, IN 46321","(MAIN QSR INC, MAIN QSR INC.)","(MAIN QSR INCORPORATED,)",2,1
2936,2937,"ONE DAY, LLC.","1417 ACADEMY, KALAMAZOO, MI 49006","(ONE DAY, LLC, ONE DAY, LLC.)","(ONE DAY, LLC,)",2,1
3449,3450,"S.T.D. MANAGEMENT, LLC","3001 10TH ST, WAYLAND, MI 49348","(S.T.D. MANAGEMENT, LLC, STD MANAGEMENT, LLC)","(STD MANAGEMENT, LLC,)",2,1


- There is 1 case of a CORP/CORPORATION merger.
- Other than that, the other 5 cases involve merging source names that differ by the inclusion/exclusion of a period.
    - For example, ONE DAY, LLC and ONE DAY, LLC.

## Suffix cases

### All people containing a suffix

In [17]:
suffix_pattern = (
    r"\b(?:JR|SR|JUNIOR|SENIOR|II|III|IV)\b"
)

suffix_entities = (
    entities[
        (entities["entity_type"] == "Person")
        &
        (
            entities["canonical_name"]
            .str.contains(
                suffix_pattern,
                regex=True,
                na=False
            )
        )
    ]
    .sort_values(
        "occurrence_count",
        ascending=False
    )
)

print(suffix_entities.shape)

suffix_entities[
    [
        "entity_id",
        "canonical_name",
        "canonical_address",
        "occurrence_count"
    ]
].head(50)

(133, 7)


,entity_id,canonical_name,canonical_address,occurrence_count
163,164,"ANDERSON, A DAVID SR & BERNADETTE","PO BOX 50204, KALAMAZOO, MI 49005-0204",12
3186,3187,"PRIMER, DANIEL L JR","6841 N SPRINKLE RD, KALAMAZOO, MI 49004",12
2148,2149,"KNOX, DANNY LEE SR","6336 RAVINE RD, KALAMAZOO, MI 49009-9002",9
1911,1912,"JACKSON, SOL JR","5370 LUCERNE AVE, KALAMAZOO, MI 49048",6
1113,1114,"DERRING III, ALBERT","3041 N 26TH ST, KALAMAZOO, MI 49048",5
4156,4157,WALL ROBERT JR,"1057 N 30TH ST, GALESBURG, MI 49053",4
604,605,"BROWN, ROBERT W. JR.","1115 DUPONT, KALAMAZOO, MI 49048-3213",4
605,606,"BROWN, ROBERT WENDELL, JR.","1115 DUPONT AVE, KALAMAZOO, MI 49048-3213",4
478,479,BOURNER RALPH B JR & THOMAS W,"6243 TWILIGHT AVE, KALAMAZOO, MI 49048",4
1538,1539,"GOOCH, MYRON EVERETT JR","1226 CADET LN, KALAMAZOO, MI 49009",4


- 146 entities contain suffixes

### Entities with both suffix and suffix-free versions

In [18]:
suffix_audit = (
    resolution_audit[
        resolution_audit[
            "entity_type_final"
        ] == "Person"
    ]
    .copy()
)

suffix_audit["has_suffix"] = (
    suffix_audit[
        "entity_name"
    ]
    .str.contains(
        suffix_pattern,
        regex=True,
        na=False
    )
)

suffix_merge_cases = (
    suffix_audit
    .groupby(
        [
            "entity_id",
            "canonical_name",
            "canonical_address"
        ]
    )
    .agg(
        source_names=(
            "entity_name",
            lambda x: tuple(
                sorted(set(x))
            )
        ),
        has_suffix_version=(
            "has_suffix",
            "any"
        ),
        has_no_suffix_version=(
            "has_suffix",
            lambda x: (~x).any()
        )
    )
    .reset_index()
)

suffix_merge_cases = (
    suffix_merge_cases[
        suffix_merge_cases["has_suffix_version"]
        &
        suffix_merge_cases["has_no_suffix_version"]
    ]
)

suffix_merge_cases.head(50)

,entity_id,canonical_name,canonical_address,source_names,has_suffix_version,has_no_suffix_version
918,1114,"DERRING III, ALBERT","3041 N 26TH ST, KALAMAZOO, MI 49048","(DERRING III, ALBERT, DERRING, ALBERT)",True,True


- There is 1 case of an entity merged due to having the same name and address, with different suffixes
    - Albert Derring and Albert Derring III

## Cases of finalized entities with the same name but different associated addresses

In [10]:
person_entities = (
    entities[
        entities["entity_type"] == "Person"
    ]
    .copy()
)

known_address_people = (
    person_entities[
        person_entities[
            "canonical_address"
        ].fillna("").str.strip() != ""
    ]
)

common_names_multiple_addresses = (
    known_address_people
    .groupby("canonical_name")
    .agg(
        entity_count=(
            "entity_id",
            "nunique"
        ),
        address_count=(
            "canonical_address",
            "nunique"
        ),
        addresses=(
            "canonical_address",
            lambda x: tuple(
                sorted(set(x))
            )
        ),
        entity_ids=(
            "entity_id",
            lambda x: tuple(
                sorted(set(x))
            )
        ),
        total_occurrences=(
            "occurrence_count",
            "sum"
        )
    )
    .reset_index()
)

common_names_multiple_addresses = (
    common_names_multiple_addresses[
        common_names_multiple_addresses[
            "address_count"
        ] > 1
    ]
    .sort_values(
        [
            "address_count",
            "total_occurrences"
        ],
        ascending=False
    )
)

print(common_names_multiple_addresses.shape)
common_names_multiple_addresses.head(50)

(293, 6)


,canonical_name,entity_count,address_count,addresses,entity_ids,total_occurrences
1087,"GATES, JEROME",4,4,"(1229 ORCHID AVE, SAINT LOUIS, MO 63147, 1229 ...","(1461, 1462, 1463, 1464)",18
2895,"TARDANI, JACOB A",4,4,"(1011 DENNER ST UNIT 1, KALAMAZOO, MI 49006, 1...","(3853, 3854, 3855, 3856)",7
2124,NEMIRE JAMES J,4,4,"(7167 SAGINAW DR, COMSTOCK, MI 49041, 7167 SAG...","(2824, 2825, 2826, 2827)",5
2620,"SANTIAGO, JORGE & MARIA",3,3,"(5101 SWALLOW AVE, KALAMAZOO, MI 49009, 5350 G...","(3499, 3500, 3501)",15
392,"BRINNEMAN, MICHAEL J",3,3,"(1802 FULFORD ST, KALAMAZOO, MI 49001, 5037 ST...","(554, 555, 556)",11
2688,"SHERROD, ANTHONY",3,3,"(1523 N WESTNEDGE AVE, KALAMAZOO, MI 49007, 36...","(3581, 3582, 3583)",8
1575,"KERNELL, BRIANT",3,3,"(10238 PORTAGE RD, PORTAGE, MI 49002, 10238 PO...","(2095, 2096, 2097)",7
3243,"WILLIAMS, LATASHA",3,3,"(1321 CAMERON ST, KALAMAZOO, MI 49001, 1321 CA...","(4310, 4311, 4312)",7
367,"BRANSON, OTHA LEE & MARY",3,3,"(514 VERNON ST, KALAMAZOO, MI 49001, 532 BRYAN...","(526, 527, 528)",6
979,"FAYLING, CHRISTOPHER M",3,3,"(2086 S 36TH ST, GALESBURG, MI 49053, 504 W VA...","(1308, 1309, 1310)",6


- There are 296 canonical_names which are associated with more than one entity/address

## Inspecting name formatting

In [11]:
people = (
    entity_occurrences[
        entity_occurrences["entity_type"] == "Person"
    ]
)

comma_names = (
    people["entity_name"]
    .str.contains(",", regex=False, na=False)
)

print(
    f"Person occurrences with comma-format names: "
    f"{comma_names.sum():,}"
)

print(
    f"Person occurrences without comma-format names: "
    f"{(~comma_names).sum():,}"
)

print(
    f"Percent parseable by current comma requirement: "
    f"{comma_names.mean():.1%}"
)

Person occurrences with comma-format names: 5,761
Person occurrences without comma-format names: 2,584
Percent parseable by current comma requirement: 69.0%
